In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway
import os
import gzip
import pickle
import json

#### Creación de funciones

In [4]:
# Carga de datos
def load_data(file_path):
    """
    Load data from a CSV file.
    """
    return pd.read_csv(file_path)

In [5]:
# Renombrar columnas
def rename_columns(df, rename_dict):
    """
    Rename columns in a DataFrame.
    """
    return df.rename(columns=rename_dict)  

In [6]:
# Eliminar registros
def delete_records(df, condition):
    """
    Delete records from a DataFrame based on a condition.
    """
    return df[~condition]

In [7]:
# Formato de datos adecuado
def format_numeric(df, column):
    """
    Convert a column to data integer.
    """
    df[column] = df[column].astype(int)
    return df

def format_date(df, column):
    """
    Convert a column to datetime.
    """
    df[column] = pd.to_datetime(df[column])
    return df

In [8]:
# Eliminar columnas
def delete_columns(df, columns):
    """
    Delete columns from a DataFrame.
    """
    return df.drop(columns=columns)

In [9]:
# Crear nuevas columnas a partir de otras
def create_new_columns(df, new_col_name,operation):
    """
    Create a new column based on existing columns.
    """
    df[new_col_name] = operation
    return df

In [10]:
def clean_outliers(df, columns):

    project_clean = df.copy()
    outlier_indices = set()  # Conjunto para almacenar los índices de las filas con valores atípicos

    for column in columns:
        # Calcular Q1, Q3 e IQR
        q1 = project_clean[column].quantile(0.25)
        q3 = project_clean[column].quantile(0.75)
        iqr = q3 - q1

        # Calcular límites superior e inferior
        upper = q3 + (2 * iqr)
        lower = q1 - (2 * iqr)

        # Identificar índices de valores atípicos
        outliers = project_clean[(project_clean[column] > upper) | (project_clean[column] < lower)].index
        outlier_indices.update(outliers)  # Agregar los índices al conjunto

    # Eliminar todas las filas con valores atípicos
    project_clean = project_clean.drop(index=outlier_indices)
    return project_clean

In [11]:
def plot_sobrecosto_distribution(df, column, title='Estado de Presupuestos'):
    """
    Genera un gráfico de barras para mostrar la distribución de proyectos con y sin sobrecosto.

    Args:
        df (pd.DataFrame): DataFrame que contiene los datos.
        column (str): Nombre de la columna que contiene los valores de sobrecosto.
        title (str): Título del gráfico. Por defecto, 'Estado de Presupuestos'.

    Returns:
        None
    """
    # Clasificar los proyectos en 'Sin Sobrecosto' y 'Con Sobrecosto'
    conteo = df[column].apply(lambda x: 'Sin Sobrecosto' if x <= 0 else 'Con Sobrecosto').value_counts()

    # Crear el gráfico de barras
    conteo.plot(kind='bar', color=['blue', 'red'], rot=0)

    # Personalizar el gráfico
    plt.title(title)
    plt.xlabel('')
    plt.ylabel('Cantidad de Proyectos')
    plt.gca().spines["top"].set_visible(False)  # Ocultar borde superior
    plt.gca().spines["right"].set_visible(False)  # Ocultar borde derecho
    plt.show()

In [12]:
def plot_histogram_subplots(df, columns, title='Distribución de Presupuestos y Gastos', bins=30, log_scale=True):
    """
    Genera subgráficos (subplots) con histogramas individuales para las columnas seleccionadas de un DataFrame.

    Args:
        df (pd.DataFrame): DataFrame que contiene los datos.
        columns (list): Lista de nombres de columnas para las que se generará el histograma.
        title (str): Título general del conjunto de gráficos.
        bins (int): Número de bins para el histograma. Por defecto, 30.
        log_scale (bool): Si True, aplica escala logarítmica al eje Y. Por defecto, True.

    Returns:
        None
    """
    n = len(columns)
    n_cols = 2  # Número de columnas de subgráficos (ajustable)
    n_rows = (n + n_cols - 1) // n_cols  # Calcular número de filas necesario

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
    axes = axes.flatten()  # Asegurar que podemos indexar los ejes fácilmente

    for i, column in enumerate(columns):
        sns.histplot(data=df, x=column, bins=bins, kde=False, ax=axes[i], color='skyblue')
        axes[i].set_title(f'{column}')
        axes[i].set_xlabel('Valores')
        axes[i].set_ylabel('Frecuencia')
        if log_scale:
            axes[i].set_yscale('log')
        axes[i].spines["top"].set_visible(False)
        axes[i].spines["right"].set_visible(False)

    # Eliminar gráficos vacíos si los hay
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title, fontsize=16)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [13]:
def plot_boxplot(df, columns, title='Distribución de Duraciones y Retrasos'):
    """
    Genera un diagrama de caja (boxplot) para las columnas seleccionadas de un DataFrame.

    Args:
        df (pd.DataFrame): DataFrame que contiene los datos.
        columns (list): Lista de nombres de columnas para las que se generará el boxplot.
        title (str): Título del gráfico. Por defecto, 'Distribución de Duraciones y Retrasos'.

    Returns:
        None
    """
    # Crear el DataFrame en formato largo para seaborn
    df_melted = df[columns].melt(var_name='Categoría', value_name='Valor')

    # Crear el diagrama de caja
    sns.catplot(x='Categoría', y='Valor', data=df_melted, kind='box')

    # Personalizar el gráfico
    plt.title(title)
    plt.xlabel('Categorías')
    plt.ylabel('Valores')
    plt.show()

In [14]:
def calcular_puntuacion_por_fase(dataframe, columna_fase, columna_probabilidad, columna_impacto, valor_fase):
    filtro = dataframe[dataframe[columna_fase] == valor_fase]
    media_probabilidad = filtro[columna_probabilidad].mean()
    media_impacto = filtro[columna_impacto].mean()
    return [media_probabilidad, media_impacto]


In [15]:
# Función para realizar ANOVA con múltiples variables categóricas y graficar resultados
def realizar_anova_con_graficos(df, variables_categoricas, target):
    """
    Realiza ANOVA para múltiples variables categóricas contra un target numérico y genera un gráfico de P-Values.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene los datos.
        variables_categoricas (list): Lista de nombres de columnas categóricas.
        target (str): Nombre de la columna numérica objetivo.
    
    Returns:
        dict: Diccionario con los resultados de ANOVA (F-statistic y p-value) para cada variable categórica.
    """
    resultados = {}
    p_values = []
    for variable in variables_categoricas:
        grupos = [df[target][df[variable] == categoria] for categoria in df[variable].unique()]
        f_stat, p_value = f_oneway(*grupos)
        resultados[variable] = {'F-Statistic': f_stat, 'P-Value': p_value}
        p_values.append(p_value)

    # Crear gráfico de línea para los P-Values
    plt.figure(figsize=(8, 5))
    plt.plot(variables_categoricas, p_values, marker='o', color='blue', label='P-Value')
    plt.axhline(0.05, color='red', linestyle='--', linewidth=0.8, label='Nivel de Significancia (0.05)')
    plt.xlabel('Variables Categóricas', fontsize=12)
    plt.ylabel('P-Value', fontsize=12)
    plt.xticks(rotation=90, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()

    return resultados

In [16]:
def correlacion_numericas_target(df, variables_numericas, target):
    """
    Calcula la correlación de Pearson entre variables numéricas y el target.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene los datos.
        variables_numericas (list): Lista de nombres de columnas numéricas.
        target (str): Nombre de la columna numérica objetivo.
    
    Returns:
        dict: Diccionario con las correlaciones de Pearson para cada variable numérica.
    """
    correlaciones = {}
    for variable in variables_numericas:
        correlacion = df[[variable, target]].corr(method='pearson').iloc[0, 1]
        correlaciones[variable] = correlacion
    return correlaciones

# Función para graficar las correlaciones
def graficar_correlaciones(correlaciones):
    """
    Genera un gráfico de barras con las correlaciones calculadas.
    
    Args:
        correlaciones (dict): Diccionario con las correlaciones de Pearson.
    
    Returns:
        None
    """
    variables = list(correlaciones.keys())
    valores = list(correlaciones.values())

    plt.figure(figsize=(6, 6))
    plt.bar(variables, valores, color='skyblue')
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    plt.xlabel('Variables Numéricas', fontsize=12)
    plt.ylabel('Coeficiente de Correlación de Pearson', fontsize=12)
    plt.xticks(rotation=90, ha='right')
    plt.tight_layout()
    plt.show()

In [17]:
def plot_feature_importance_from_file(model_path):
    """
    Genera un gráfico de barras mostrando la importancia de las características
    de un modelo entrenado cargado desde un archivo.

    Args:
        model_path (str): Ruta al archivo del modelo entrenado (.pkl.gz).
    """
    # Cargar el modelo entrenado
    with gzip.open(model_path, "rb") as file:
        trained_model = pickle.load(file)

    # Verificar si el modelo tiene el atributo 'feature_importances_' o 'coef_'
    if hasattr(trained_model.best_estimator_.named_steps["regressor"], "feature_importances_"):
        # Modelos basados en árboles
        feature_importances = trained_model.best_estimator_.named_steps["regressor"].feature_importances_
    elif hasattr(trained_model.best_estimator_.named_steps["regressor"], "coef_"):
        # Modelos lineales
        feature_importances = trained_model.best_estimator_.named_steps["regressor"].coef_
    else:
        raise AttributeError("El modelo no tiene 'feature_importances_' ni 'coef_'.")

    # Obtener los nombres de las características
    preprocessor = trained_model.best_estimator_.named_steps["preprocessor"]
    categorical_features = preprocessor.transformers_[1][2]
    numerical_features = preprocessor.transformers_[0][2]

    # Obtener nombres codificados para variables categóricas
    encoded_categorical_features = preprocessor.transformers_[1][1].get_feature_names_out(categorical_features)

    # Combinar nombres de características numéricas y categóricas
    all_feature_names = list(numerical_features) + list(encoded_categorical_features)

    # Crear un DataFrame para la importancia de las características
    importance_df = pd.DataFrame({
        "Feature": all_feature_names,
        "Importance": feature_importances
    }).sort_values(by="Importance", ascending=False)

    # Graficar la importancia de las características
    plt.figure(figsize=(10, 6))
    sns.barplot(x="Importance", y="Feature", data=importance_df, palette="viridis")
    plt.title("Importancia de las Características")
    plt.xlabel("Importancia")
    plt.ylabel("Características")
    plt.tight_layout()
    plt.show()

In [18]:
def plot_model_predictions(model_path, x_test, y_test):
    """
    Carga un modelo entrenado desde un archivo .pkl.gz y genera un gráfico de predicciones vs valores reales.

    Args:
        model_path (str): Ruta del archivo del modelo entrenado (.pkl.gz).
        x_test (pd.DataFrame): Conjunto de características de prueba.
        y_test (pd.Series): Valores reales del conjunto de prueba.

    Returns:
        None
    """
    # Cargar el modelo desde el archivo .pkl.gz
    with gzip.open(model_path, 'rb') as file:
        model = pickle.load(file)

    # Realizar predicciones
    y_test_pred = model.predict(x_test)

    # Crear el gráfico
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_test_pred, alpha=0.3, label="Predicciones")
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label="Línea Ideal")
    plt.xlabel('Valores Reales')
    plt.ylabel('Predicciones')
    plt.title('Predicciones vs Valores Reales')
    plt.legend()
    plt.show()

In [19]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    median_absolute_error)
from sklearn.model_selection import train_test_split

def load_model_from_gz(file_path):
    with gzip.open(file_path, 'rb') as f:
        model = pickle.load(f)
    return model

def metrics_calculate_from_file(model_path, x_train, x_test, y_train, y_test, output_path):
    # Cargar el modelo
    model = load_model_from_gz(model_path)

    # Realizar predicciones
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # Calcular métricas
    metrics = [
        {
            'type': 'metrics',
            'dataset': 'train',
            'r2': r2_score(y_train, y_train_pred),
            'mse': mean_squared_error(y_train, y_train_pred),
            'mad': median_absolute_error(y_train, y_train_pred),
        },
        {
            'type': 'metrics',
            'dataset': 'test',
            'r2': r2_score(y_test, y_test_pred),
            'mse': mean_squared_error(y_test, y_test_pred),
            'mad': median_absolute_error(y_test, y_test_pred),
        }
    ]

    # Crear directorio de salida y guardar resultados
    os.makedirs('files/output', exist_ok=True)
    with open(output_path, 'w') as file:
        for metric in metrics:
            file.write(json.dumps(metric) + '\n')

#### Proceso de limpieza

In [ ]:
project=load_data('Archivos/Data_input/Capital_Project_Schedules_and_Budgets.csv')

In [ ]:
# estructura de datos
project.info()

In [ ]:
# Renombrar columnas
rename_dict={
    'Project Geographic District ':'Distrito Geografico',
    'Project Building Identifier':'ID Edificio',
    'Project School Name':'Nombre Escuela',
    'Project Type ':'Tipo de Proyecto',
    'Project Description':'Descripción del Proyecto',
    'Project Phase Name':'Fase del Proyecto',
    'Project Status Name':'Estado del Proyecto',
    'Project Phase Actual Start Date':'Fecha de Inicio',
    'Project Phase Planned End Date':'Fecha de Fin Proyectada',
    'Project Phase Actual End Date':'Fecha de Fin Real',
    'Project Budget Amount':'Presupuesto del Proyecto',
    'Final Estimate of Actual Costs Through End of Phase Amount':'Presupuesto Final Estimado',
    'Total Phase Actual Spending Amount':'Cantidad Real Gastada',
    'DSF Number(s)':'Identificador DSF'
}
project=rename_columns(project, rename_dict)

In [ ]:
# Eliminar registros con valores incoherentes
filtro1=project['Presupuesto del Proyecto'].isin(['DOER', '0', 'DIIR', 'DOES', 'IEH', 'FTK', 'EMER', 'DIIT', 'DOEL', 'TPL', 'DOEP'])
filtro2=project['Estado del Proyecto'].isin(['PNS'])

project=delete_records(project, filtro1)
project=delete_records(project, filtro2)


In [ ]:
# Cambiar formato de variables
project=format_date(project, 'Fecha de Inicio')
project=format_date(project, 'Fecha de Fin Proyectada')
project=format_date(project, 'Fecha de Fin Real')
project=format_numeric(project, 'Presupuesto del Proyecto')

In [ ]:
# Eliminar columnas innecesarias
project=delete_columns(project, columns=['Identificador DSF', 'ID Edificio'])

In [ ]:
project.to_excel('Archivos/Capital_Project_Schedules_and_Budgets_2024.xlsx', index=False)

In [ ]:
# Creación de nuevas columnas a partir de las existentes
project=create_new_columns(project,'Duración Proyectada',
                            (project['Fecha de Fin Proyectada']-project['Fecha de Inicio']).dt.days)

project=create_new_columns(project,'Duración Real',
                            (project['Fecha de Fin Real']-project['Fecha de Inicio']).dt.days)

project=create_new_columns(project,'Retraso',
                            (project['Fecha de Fin Real']-project['Fecha de Fin Proyectada']).dt.days)

project=create_new_columns(project,'Desviación Presupuestaria',
                           round(((project['Cantidad Real Gastada']-project['Presupuesto del Proyecto'])/project['Presupuesto del Proyecto'])*100,2))


In [ ]:
# Eliminar columnas innecesarias
project=delete_columns(project, columns=['Fecha de Fin Real', 'Fecha de Fin Proyectada', 'Fecha de Inicio'])

In [ ]:
# Eliminar proyectos con duración real cero o negativa y con cantidad real gastada cero o negativa
filtro3=project['Duración Real']<=0
filtro4=project['Cantidad Real Gastada']<=0
project=delete_records(project, filtro3)
project=delete_records(project, filtro4)

In [ ]:
# dividir la base de datos en dos partes: proyectos completos y en progreso
# Proyectos completos
project_complete=project[project['Estado del Proyecto']=='Complete']
project_complete.dropna(inplace=True)
# Proyectos en progreso
project_inprogress=project[project['Estado del Proyecto']=='In-Progress']
project_inprogress=delete_columns(project_inprogress, columns=['Duración Real','Retraso'])
project_inprogress.dropna(inplace=True)

In [ ]:
# Guardar los DataFrames
project_complete.to_csv('Archivos/Data_output/Capital_Project_Schedules_and_Budgets_clean_complete.csv', index=False)
project_inprogress.to_csv('Archivos/Data_output/Capital_Project_Schedules_and_Budgets_clean_inprogress.csv', index=False)

#### Trabajo con DataFrame_Complete

In [ ]:
project_complete=load_data('Archivos/Data_output/Capital_Project_Schedules_and_Budgets_clean_complete.csv')

In [ ]:
plot_sobrecosto_distribution(project_complete, 'Desviación Presupuestaria')

In [ ]:
project_complete.describe()

In [ ]:
# # histograma de presupuestos y gastos
# plot_histogram_subplots(project_complete,
#                        ['Presupuesto del Proyecto','Presupuesto Final Estimado','Cantidad Real Gastada'], 
#                        title='Distribución de Presupuestos y Gastos', 
#                        bins=30, 
#                        log_scale=True)

In [ ]:
# # boxplot de duraciones y retrasos
# plot_boxplot(project_complete, 
#              ['Duración Proyectada', 'Duración Real', 'Retraso'], 
#              title='Distribución de Duraciones y Retrasos')

In [ ]:
project_clean_complete=clean_outliers(project_complete, columns=['Presupuesto del Proyecto', 'Presupuesto Final Estimado', 'Cantidad Real Gastada', 
                                                                'Duración Proyectada', 'Duración Real', 'Retraso','Desviación Presupuestaria'])
project_clean_complete=project_clean_complete[~project_clean_complete['Presupuesto Final Estimado'].isin([28,186,176,179,164,154])]
project_clean_complete=project_clean_complete[~project_clean_complete['Duración Real'].isin(range(1,30))]
project_clean_complete=project_clean_complete[~project_clean_complete['Cantidad Real Gastada'].isin([6,15,59,74])]
project_clean_complete=delete_records(project_clean_complete, project_clean_complete['Fase del Proyecto']=='Construction')

In [ ]:
project_clean_complete.describe()

In [ ]:
# plot_histogram_subplots(project_clean_complete,
#                        ['Presupuesto del Proyecto','Presupuesto Final Estimado','Cantidad Real Gastada'], 
#                        title='Distribución de Presupuestos y Gastos', 
#                        bins=30, 
#                        log_scale=True)

In [ ]:
# # boxplot de duraciones y retrasos
# plot_boxplot(project_clean_complete, 
#              ['Duración Proyectada', 'Duración Real', 'Retraso'], 
#              title='Distribución de Duraciones y Retrasos')

In [ ]:
sns.histplot(data=project_clean_complete, x='Desviación Presupuestaria', kde=True, bins=30)
plt.title('Distribución de Desviación Presupuestaria')
plt.xlabel('Porcentaje de Desviación')
plt.ylabel('Frecuencia')
plt.gca().spines["top"].set_visible(False)  # Ocultar borde superior
plt.gca().spines["right"].set_visible(False)  # Ocultar borde derecho
plt.show()

In [ ]:
# palette = {
#     'Scope': 'blue',
#     'Design': 'green',
#     'CM,F&E': 'orange'
# }
# sns.catplot(x='Fase del Proyecto', y='Sobrecosto', data=project_clean_complete, kind='box', palette=palette, height=6, aspect=2)
# plt.title('Desviación Presupuestaria por Fase del Proyecto')
# plt.xlabel('Fase del Proyecto')
# plt.ylabel('Sobrecosto (%)')
# plt.show()

In [ ]:
# #organizar columnas
# column_order=['Distrito Geografico', 'Nombre Escuela', 'Tipo de Proyecto',
#        'Descripción del Proyecto', 'Fase del Proyecto', 'Estado del Proyecto',
#        'Presupuesto del Proyecto', 'Presupuesto Final Estimado',
#        'Desviación Presupuestaria','Duración Proyectada', 'Duración Real',
#        'Retraso', 'Cantidad Real Gastada']
# project_clean_complete=project_clean_complete[column_order]

# filtered_data = project_clean_complete.select_dtypes('number')

# # Calcular la matriz de correlación
# correlation_matrix = filtered_data.corr()

# # Crear el heatmap
# plt.figure(figsize=(10, 8))
# sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
# plt.title("Heatmap de Correlación")
# plt.show()

In [ ]:
#crear la columna categoria proyecto a partir de la descripcion del proyecto

#definir las categorias y palabras clave
categoria_palabras = {
    "Mejoras exteriores": ['exterior', 'masonry', 'roofs', 'parapets', 'windows', 'replacement'],
    "Sistemas de seguridad": ['fire', 'alarm', 'camera', 'system', 'ipdvs', 'installations'],
    "Control climático": ['climate', 'ventilation', 'boiler', 'freezers', 'electrical', 'low', 'voltage'],
    "Infraestructura escolar": ['auditorium', 'playground', 'redevelopment', 'walk', 'accessibility'],
    "Proyectos ambientales": ['flood', 'elimination', 'planyc'],
    "Transformaciones internas": ['conversion', 'upgrade', 'installation', 'systems'],
    "COVID / Salud": ['covid', 'ida'],
    "Transporte / movilidad": ['path', 'travel'],
    "Programas educativos": ['program', 'full', 'reso']
}

def asignar_categoria(descripcion):
    descripcion_lower = str(descripcion).lower()
    for categoria, palabras in categoria_palabras.items():
        if any(palabra in descripcion_lower for palabra in palabras):
            return categoria
    return 'Otro'

project_clean_complete['Categoria Proyecto'] = project_clean_complete['Descripción del Proyecto'].apply(asignar_categoria)

# eliminar columna descripcion del proyecto
project_clean_complete=delete_columns(project_clean_complete, columns=['Descripción del Proyecto'])

In [ ]:
#Renombrar los tipos de proyecto para que sean más descriptivos
renombrar_tipo_proyecto = {
    "SCA CIP": "Proyecto de Mejoramiento General (CIP)",
    "SCA CIP RESOA": "Mejoras Asociadas a Programas Académicos (RESOA)",
    "3K": "Adecuaciones para Educación Inicial 3K",
    "SCA Lease Site Improvement": "Mejoras en Sedes Arrendadas",
    "SCA Capacity": "Expansión de Capacidad Escolar",
    "PRE-K": "Adecuaciones para Educación Preescolar (Pre-K)",
    "SCA Emergency Lighting": "Instalación de Iluminación de Emergencia"
}

# Crear nueva columna con nombres más descriptivos
project_clean_complete["Tipo Proyecto"] = project_clean_complete["Tipo de Proyecto"].map(renombrar_tipo_proyecto).fillna("Otro")
# eliminar columna tipo de proyecto
project_clean_complete=delete_columns(project_clean_complete, columns=['Tipo de Proyecto'])

#organizar columnas en el orden deseado
column_order=['Distrito Geografico', 'Nombre Escuela', 'Tipo Proyecto',
       'Categoria Proyecto', 'Fase del Proyecto', 'Estado del Proyecto',
       'Presupuesto del Proyecto', 'Presupuesto Final Estimado',
       'Desviación Presupuestaria','Duración Proyectada', 'Duración Real',
       'Retraso', 'Cantidad Real Gastada']
project_clean_complete=project_clean_complete[column_order]

In [ ]:
sns.pairplot(data=project_clean_complete, vars=['Presupuesto del Proyecto', 'Presupuesto Final Estimado', 'Cantidad Real Gastada',
                                                'Duración Proyectada', 'Duración Real', 'Retraso','Desviación Presupuestaria'])
plt.show()

In [ ]:
project_clean_complete.to_csv('Archivos/Data_output_transform/Project_complete.csv', index=False)

In [ ]:
# # proyectos que gastaron más de lo presupuestado y terminaron antes de tiempo                       
# project_clean_complete[(project_clean_complete['Cantidad Real Gastada']>project_clean_complete['Presupuesto del Proyecto'])&
#                        (project_clean_complete['Retraso']<0)
#                        &(project_clean_complete['Desviación Presupuestaria']>0)]

# # proyectos que gastaron menos de lo presupuestado y terminaron después de tiempo
# project_clean_complete[(project_clean_complete['Cantidad Real Gastada']<project_clean_complete['Presupuesto del Proyecto'])&
#                        (project_clean_complete['Retraso']>0)
#                        &(project_clean_complete['Desviación Presupuestaria']<0)]

# #proyectos que gastaron más de lo presupuestado y terminaron después de tiempo
# project_clean_complete[(project_clean_complete['Cantidad Real Gastada']>project_clean_complete['Presupuesto del Proyecto'])&
#                        (project_clean_complete['Retraso']>0)
#                        &(project_clean_complete['Desviación Presupuestaria']>0)]

# #proyectos que gastaron menos de lo presupuestado y terminaron antes de tiempo
# project_clean_complete[(project_clean_complete['Cantidad Real Gastada']<project_clean_complete['Presupuesto del Proyecto'])&
#                         (project_clean_complete['Retraso']<0)
#                         &(project_clean_complete['Desviación Presupuestaria']<0)]

In [ ]:
project_clean_complete.columns

#### Trabajo con DataFrame_In-Progress limpio

In [ ]:
project_inprogress=load_data('Archivos/Data_output/Capital_Project_Schedules_and_Budgets_clean_inprogress.csv')

In [ ]:
project_clean_progress=clean_outliers(project_inprogress, columns=['Presupuesto del Proyecto', 'Presupuesto Final Estimado', 'Cantidad Real Gastada', 
                                                                'Duración Proyectada','Desviación Presupuestaria'])
project_clean_progress=project_clean_progress[~project_clean_progress['Duración Proyectada'].isin([34,59,89,90,91,92])]
project_clean_progress=project_clean_progress[~project_clean_progress['Cantidad Real Gastada'].isin([36,53,112,160,169])]
project_clean_progress=project_clean_progress[project_clean_progress['Desviación Presupuestaria']<-10]

In [ ]:
project_clean_progress.describe()

In [ ]:
plot_histogram_subplots(project_clean_progress,
                       ['Presupuesto del Proyecto','Presupuesto Final Estimado','Cantidad Real Gastada'], 
                       title='Distribución de Presupuestos y Gastos', 
                       bins=30, 
                       log_scale=True)

In [ ]:
filtered_data = project_clean_progress.select_dtypes('number')

# Calcular la matriz de correlación
correlation_matrix = filtered_data.corr()

# Crear el heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Heatmap de Correlación")
plt.show()

In [ ]:
#crear la columna categoria proyecto a partir de la descripcion del proyecto

#definir las categorias y palabras clave
categoria_palabras = {
    "Mejoras exteriores": ['exterior', 'masonry', 'roofs', 'parapets', 'windows', 'replacement'],
    "Sistemas de seguridad": ['fire', 'alarm', 'camera', 'system', 'ipdvs', 'installations'],
    "Control climático": ['climate', 'ventilation', 'boiler', 'freezers', 'electrical', 'low', 'voltage'],
    "Infraestructura escolar": ['auditorium', 'playground', 'redevelopment', 'walk', 'accessibility'],
    "Proyectos ambientales": ['flood', 'elimination', 'planyc'],
    "Transformaciones internas": ['conversion', 'upgrade', 'installation', 'systems'],
    "COVID / Salud": ['covid', 'ida'],
    "Transporte / movilidad": ['path', 'travel'],
    "Programas educativos": ['program', 'full', 'reso']
}

def asignar_categoria(descripcion):
    descripcion_lower = str(descripcion).lower()
    for categoria, palabras in categoria_palabras.items():
        if any(palabra in descripcion_lower for palabra in palabras):
            return categoria
    return 'Otro'

project_clean_progress['Categoria Proyecto'] = project_clean_progress['Descripción del Proyecto'].apply(asignar_categoria)

# eliminar columna descripcion del proyecto
project_clean_progress=delete_columns(project_clean_progress, columns=['Descripción del Proyecto'])

In [ ]:

#Renombrar los tipos de proyecto para que sean más descriptivos
renombrar_tipo_proyecto = {
    "SCA CIP": "Proyecto de Mejoramiento General (CIP)",
    "SCA CIP RESOA": "Mejoras Asociadas a Programas Académicos (RESOA)",
    "3K": "Adecuaciones para Educación Inicial 3K",
    "SCA Lease Site Improvement": "Mejoras en Sedes Arrendadas",
    "SCA Capacity": "Expansión de Capacidad Escolar",
    "PRE-K": "Adecuaciones para Educación Preescolar (Pre-K)",
    "SCA Emergency Lighting": "Instalación de Iluminación de Emergencia"
}

# Crear nueva columna con nombres más descriptivos
project_clean_progress["Tipo Proyecto"] = project_clean_progress["Tipo de Proyecto"].map(renombrar_tipo_proyecto).fillna("Otro")
# eliminar columna tipo de proyecto
project_clean_progress=delete_columns(project_clean_progress, columns=['Tipo de Proyecto'])

#organizar columnas en el orden deseado
column_order=['Distrito Geografico', 'Nombre Escuela', 'Tipo Proyecto',
       'Categoria Proyecto', 'Fase del Proyecto', 'Estado del Proyecto',
       'Presupuesto del Proyecto', 'Presupuesto Final Estimado',
       'Desviación Presupuestaria','Duración Proyectada',
       'Cantidad Real Gastada']
project_clean_progress=project_clean_progress[column_order]

project_clean_progress=rename_columns(project_clean_progress, {'Cantidad Real Gastada':'Cantidad Gastada a la Fecha'})

In [ ]:
project_clean_progress.to_csv('Archivos/Data_output_transform/Project_progress.csv', index=False)

In [ ]:
project_clean_progress.columns

#### Creación de features y el dataframe de trabajo

In [ ]:
project=load_data('Archivos/Data_output_transform/Project_complete.csv')
project_2=load_data('Archivos/Data_output_transform/Project_progress.csv')
prob_imp=pd.read_csv('Archivos/Data_input/Probabilidad_Impacto.csv', sep=';')

In [ ]:
# Creación de dos nuevas variables: Probabilidad e Impacto

# Calular la puntuación de probabilidad e impacto por fase, retraso y presupuesto
result1 = calcular_puntuacion_por_fase(prob_imp, 'Fase', 'Media de Probabilidad', 'Impacto Medio', 'Scope')
result2 = calcular_puntuacion_por_fase(prob_imp, 'Fase', 'Media de Probabilidad', 'Impacto Medio', 'Design')
result3 = calcular_puntuacion_por_fase(prob_imp, 'retraso >30%', 'Media de Probabilidad', 'Impacto Medio', 'Retraso')
result4 = calcular_puntuacion_por_fase(prob_imp, 'rango 1', 'Media de Probabilidad', 'Impacto Medio', 'Presupuesto')
result5 = calcular_puntuacion_por_fase(prob_imp, 'rango 2', 'Media de Probabilidad', 'Impacto Medio', 'Presupuesto')
result6 = calcular_puntuacion_por_fase(prob_imp, 'rango 3', 'Media de Probabilidad', 'Impacto Medio', 'Presupuesto')
result7 = calcular_puntuacion_por_fase(prob_imp, 'rango 4', 'Media de Probabilidad', 'Impacto Medio', 'Presupuesto')

# Filtrar los valores donde 'Fase del Proyecto' sea 'Scope' y asignar result1
project.loc[project['Fase del Proyecto'] == 'Scope', 'ScopeP'] = result1[0]
project.loc[project['Fase del Proyecto'] == 'Scope', 'ScopeI'] = result1[1]

# Filtrar los valores donde 'Fase del Proyecto' sea 'Design' y asignar result2
project.loc[project['Fase del Proyecto'] == 'Design', 'DesignP'] = result2[0]
project.loc[project['Fase del Proyecto'] == 'Design', 'DesignI'] = result2[1]

# Filtrar los valores donde 'Retraso' sea 'retraso >30%' y asignar result3
project.loc[project['Retraso'] >= 0.3*project['Duración Proyectada'], 'RetrasoP'] = result3[0]
project.loc[project['Retraso'] >= 0.3*project['Duración Proyectada'], 'RetrasoI'] = result3[1]

# Filtrar los valores donde 'Fase del Proyecto' sea 'rango 1' y asignar result4
percentil_25 = project['Presupuesto del Proyecto'].quantile(0.25)
project.loc[project['Presupuesto del Proyecto'] <= percentil_25, 'rango1P'] = result4[0]
project.loc[project['Presupuesto del Proyecto'] <= percentil_25, 'rango1I'] = result4[1]

# Filtrar los valores donde 'Fase del Proyecto' sea 'rango 2' y asignar result5
percentil_75 = project['Presupuesto del Proyecto'].quantile(0.75)
project.loc[(project['Presupuesto del Proyecto'] > percentil_25) & (project['Presupuesto del Proyecto'] <= percentil_75), 'rango2P'] = result5[0]
project.loc[(project['Presupuesto del Proyecto'] > percentil_25) & (project['Presupuesto del Proyecto'] <= percentil_75), 'rango2I'] = result5[1]

# Filtrar los valores donde 'Fase del Proyecto' sea 'rango 3' y asignar result6
percentil_90 = project['Presupuesto del Proyecto'].quantile(0.90)
project.loc[(project['Presupuesto del Proyecto'] > percentil_75) & (project['Presupuesto del Proyecto'] <= percentil_90), 'rango3P'] = result6[0]
project.loc[(project['Presupuesto del Proyecto'] > percentil_75) & (project['Presupuesto del Proyecto'] <= percentil_90), 'rango3I'] = result6[1]

# Filtrar los valores donde 'Fase del Proyecto' sea 'rango 4' y asignar result7
project.loc[project['Presupuesto del Proyecto'] > percentil_90, 'rango4P'] = result7[0]
project.loc[project['Presupuesto del Proyecto'] > percentil_90, 'rango4I'] = result7[1]

#Creación de nuevas columnas a partir de las existentes
project['Probabilidad']=project[['ScopeP','DesignP','RetrasoP','rango1P','rango2P','rango3P','rango4P']].mean(axis=1)
project['Impacto']=project[['ScopeI','DesignI','RetrasoI','rango1I','rango2I','rango3I','rango4I']].mean(axis=1)

#Eliminar columnas innecesarias
project=delete_columns(project, columns=['ScopeP','DesignP','RetrasoP','rango1P','rango2P','rango3P','rango4P',
                                         'ScopeI','DesignI','RetrasoI','rango1I','rango2I','rango3I','rango4I'])

In [ ]:
# # Creación de modelo para estimar gasto a la fecha en proyectos completados

# from sklearn.ensemble import RandomForestRegressor
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import train_test_split

# df_train = project_2.copy()

# # Entrenamiento del modelo con datos de proyectos en progreso
# X = df_train[['Tipo Proyecto',
#        'Categoria Proyecto', 'Fase del Proyecto',
#        'Presupuesto del Proyecto', 'Presupuesto Final Estimado',
#        'Desviación Presupuestaria', 'Duración Proyectada']]
# y = df_train['Cantidad Gastada a la Fecha']

# # División de datos para prueba/entrenamiento
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Codificación de variables categóricas
# categorical_cols = ['Tipo Proyecto', 'Categoria Proyecto', 'Fase del Proyecto']
# numeric_cols = ['Presupuesto del Proyecto', 'Presupuesto Final Estimado',
#                 'Desviación Presupuestaria', 'Duración Proyectada']

# preprocessor = ColumnTransformer([
#     ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
# ], remainder='passthrough')

# # Pipeline de entrenamiento
# modelo_gasto_a_fecha = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
# ])

In [ ]:
# from sklearn.metrics import mean_absolute_error, r2_score
# from sklearn.model_selection import train_test_split

# # entrenar modelo en datos de entrenamiento
# modelo_gasto_a_fecha.fit(X_train, y_train)

# # Predecir en test
# y_pred = modelo_gasto_a_fecha.predict(X_test)

# # Calcular métricas
# mae = mean_absolute_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# print(f"MAE:  ${mae:,.0f}")
# print(f"R²:   {r2:.2f}")

In [ ]:
# plt.scatter(y_test, y_pred, alpha=0.7)
# plt.xlabel("Gasto simulado real")
# plt.ylabel("Gasto simulado predicho")
# plt.title("Comparación real vs predicho")
# plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
# plt.grid(True)
# plt.show()

In [ ]:
# #Aplicar el modelo para generar la columna "gasto_a_la_fecha_estimado" en proyectos completados
# X_pred = project[['Tipo Proyecto',
#        'Categoria Proyecto', 'Fase del Proyecto',
#        'Presupuesto del Proyecto', 'Presupuesto Final Estimado',
#        'Desviación Presupuestaria', 'Duración Proyectada']]
# project['Gasto a la Fecha Estimado'] = modelo_gasto_a_fecha.predict(X_pred)

# #organizar columnas en el orden deseado
# column_order=['Tipo Proyecto',
#        'Categoria Proyecto', 'Fase del Proyecto',
#        'Presupuesto del Proyecto', 'Presupuesto Final Estimado',
#        'Desviación Presupuestaria','Duración Proyectada',
#        'Gasto a la Fecha Estimado','Probabilidad','Impacto',
#        'Cantidad Real Gastada']
# project=project[column_order]

In [ ]:
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler

# steps=[('preprocessor', ColumnTransformer(
#   transformers=[
#       ('num', StandardScaler(), ['impacto','Presupuesto del Proyecto','probabilidad',
#                        'Duración Proyectada','Presupuesto Final Estimado','Cantidad Real Gastada']),
#       ('cat', OneHotEncoder(), ['Fase del Proyecto','Estado del Proyecto','Tipo de Proyecto'])
#   ]  
# ))
# ]


In [ ]:
# output_dir = os.path.join("Archivos", "Data")
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)

# project.to_csv(os.path.join(output_dir, "DataFrame.csv"), index=False)

In [ ]:
data=load_data('Archivos/Data/DataFrame.csv')

In [ ]:
filtered_data = data[['Presupuesto del Proyecto', 'Presupuesto Final Estimado','Desviación Presupuestaria',
                 'Duración Proyectada','Probabilidad','Impacto','Gasto a la Fecha Estimado',
                 'Cantidad Real Gastada']]

# Calcular la matriz de correlación
correlation_matrix = filtered_data.corr()

# Crear el heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Heatmap de Correlación")
plt.show()

In [ ]:
# sns.pairplot(data=data, vars=['Presupuesto del Proyecto', 'Presupuesto Final Estimado','Desviación Presupuestaria',
#                   'Duración Proyectada','Probabilidad','Impacto','Gasto a la Fecha Estimado',
#                   'Cantidad Real Gastada'])
# plt.show()

In [ ]:
correlaciones=correlacion_numericas_target(data, variables_numericas=['Gasto a la Fecha Estimado', 'Presupuesto Final Estimado',
                                                                      'Presupuesto del Proyecto','Probabilidad','Impacto','Duración Proyectada',
                                                                      'Desviación Presupuestaria']
                                                                    , target='Cantidad Real Gastada')
graficar_correlaciones(correlaciones)

In [ ]:
realizar_anova_con_graficos(data, variables_categoricas=['Tipo Proyecto','Categoria Proyecto','Fase del Proyecto'], 
                            target='Cantidad Real Gastada')

#### Regresión Lineal


In [ ]:
# from sklearn.linear_model import LinearRegression
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.feature_selection import SelectKBest, f_regression
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import GridSearchCV

In [ ]:
# lectura de la data
project_2025=pd.read_csv('Archivos/Data/DataFrame.csv')

In [ ]:
# # preprocesamiento de la data
# numerical_features=['Gasto a la Fecha Estimado', 'Presupuesto Final Estimado',
#                     'Presupuesto del Proyecto','Probabilidad','Impacto','Duración Proyectada',
#                     'Desviación Presupuestaria']
# categorical_features=['Tipo Proyecto','Fase del Proyecto']

# preprocessor=ColumnTransformer(
#     transformers=
#     [('num', StandardScaler(),numerical_features),
#     ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
#     ])

# # creación de la pipeline
# steps=[
#     ('preprocessor', preprocessor),
#     ('feature_selection', SelectKBest(f_regression)),
#     ('regressor', LinearRegression())
# ]
# pipeline=Pipeline(steps)

In [ ]:
# # hiperparámetros para la búsqueda de cuadrícula
# param_grid={
#     'feature_selection__k': [5, 10, 15, 17],
#     'regressor__fit_intercept': [True, False]
# }

# # búsqueda de cuadrícula
# model=GridSearchCV(pipeline,
#                    param_grid=param_grid,
#                    cv=10,
#                    scoring='neg_mean_squared_error',
#                    n_jobs=-1,
#                    refit=True)

In [ ]:
# partición de la data en conjunto de entrenamiento y prueba
x_train, x_test, y_train, y_test=train_test_split(
    project_2025.drop(columns=['Cantidad Real Gastada']),
    project_2025['Cantidad Real Gastada'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# # ajuste del modelo
# model.fit(x_train, y_train)
# print("Best score: ", model.best_score_)
# print("Best parameters: ", model.best_params_)

In [ ]:
# # alamacenamiento del modelo
# model_filename='files/models'
# os.makedirs(model_filename,exist_ok=True)

# model_path=os.path.join(model_filename,'RegresionLineal.pkl.gz')
# with gzip.open(model_path,'wb') as file:
#     pickle.dump(model,file)

In [ ]:
model_path='files/models/RegresionLineal.pkl.gz'

In [ ]:
# caculo de metricas del modelo
metrics_calculate_from_file(model_path, x_train, x_test, y_train, y_test,
                            output_path='files/output/metrics1.json')

In [ ]:
# Grafico de ajuste del modelo
plot_model_predictions(model_path, x_test, y_test)

#### Random Forest Regressor


In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import train_test_split
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import (
#     r2_score,
#     mean_squared_error,
#     median_absolute_error)

In [ ]:
# lectura de la data
project_2025=pd.read_csv('Archivos/Data/DataFrame.csv')

In [ ]:
# # preprocesamiento de la data
# numerical_features=['Gasto a la Fecha Estimado', 'Presupuesto Final Estimado',
#                     'Presupuesto del Proyecto','Probabilidad','Impacto','Duración Proyectada',
#                     'Desviación Presupuestaria']
# categorical_features=['Tipo Proyecto','Fase del Proyecto']

# preprocessor=ColumnTransformer(
#     transformers=
#     [('num', StandardScaler(),numerical_features),
#     ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
#     ])

# # creación de la pipeline
# steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor())
# ]
# pipeline=Pipeline(steps)

In [ ]:
# # hiperparámetros para la búsqueda de cuadrícula
# param_grid={
#     'regressor__n_estimators': [100, 200, 300],  # Número de árboles en el bosque
#     'regressor__max_depth': [10, 20, 30, None],  # Profundidad máxima de los árboles
#     'regressor__min_samples_split': [2, 5, 10],  # Mínimo número de muestras para dividir un nodo
#     'regressor__min_samples_leaf': [1, 2, 4],  # Mínimo número de muestras en una hoja
#     'regressor__max_features': ['auto', 'sqrt', 'log2'],  # Número de características a considerar en cada división
#     'regressor__bootstrap': [True, False]  # Si se utiliza muestreo con reemplazo
# }

# # búsqueda de cuadrícula
# model=GridSearchCV(pipeline,
#                    param_grid=param_grid,
#                    cv=10,
#                    scoring='neg_mean_squared_error',
#                    n_jobs=-1,
#                    refit=True)

In [ ]:
# partición de la data en conjunto de entrenamiento y prueba
x_train, x_test, y_train, y_test=train_test_split(
    project_2025.drop(columns=['Cantidad Real Gastada']),
    project_2025['Cantidad Real Gastada'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# # ajuste del modelo
# model.fit(x_train, y_train)
# print("Best score: ", model.best_score_)
# print("Best parameters: ", model.best_params_)

In [ ]:
# # alamacenamiento del modelo
# model_filename='files/models'
# os.makedirs(model_filename,exist_ok=True)

# model_path=os.path.join(model_filename,'ArbolesdeDecision.pkl.gz')
# with gzip.open(model_path,'wb') as file:
#     pickle.dump(model,file)

In [ ]:
model_path='files/models/ArbolesdeDecision.pkl.gz'

In [ ]:
# caculo de metricas del modelo
metrics_calculate_from_file(model_path, x_train, x_test, y_train, y_test,
                            output_path='files/output/metrics2.json')

In [ ]:
# Grafico de ajuste del modelo
plot_model_predictions(model_path, x_test, y_test)

In [ ]:
# importancia de las variables del modelo
plot_feature_importance_from_file(model_path)

#### Gradient Boosting Regressor


In [ ]:
# from sklearn.ensemble import GradientBoostingRegressor
# from sklearn.model_selection import train_test_split
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import (
#     r2_score,
#     mean_squared_error,
#     median_absolute_error)

In [ ]:
# lectura de la data
project_2025=pd.read_csv('Archivos/Data/DataFrame.csv')

In [ ]:
# # preprocesamiento de la data
# numerical_features=['Gasto a la Fecha Estimado', 'Presupuesto Final Estimado',
#                     'Presupuesto del Proyecto','Probabilidad','Impacto','Duración Proyectada',
#                     'Desviación Presupuestaria']
# categorical_features=['Tipo Proyecto','Fase del Proyecto']

# preprocessor=ColumnTransformer(
#     transformers=
#     [('num', StandardScaler(),numerical_features),
#     ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
#     ])

# # creación de la pipeline
# steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', GradientBoostingRegressor())
# ]
# pipeline=Pipeline(steps)

In [ ]:
# # hiperparámetros para la búsqueda de cuadrícula
# param_grid={
#     'regressor__n_estimators': [100, 200, 300],  # Número de árboles en el modelo
#     'regressor__learning_rate': [0.01, 0.05, 0.1],  # Tasa de aprendizaje
#     'regressor__max_depth': [3, 5, 7],  # Profundidad máxima de los árboles
#     'regressor__min_samples_split': [2, 5, 10],  # Mínimo de muestras para dividir un nodo
#     'regressor__min_samples_leaf': [1, 2, 4],  # Mínimo de muestras en una hoja
#     'regressor__subsample': [0.8, 1.0],  # Fracción de muestras utilizadas para entrenar cada árbol
#     'regressor__max_features': ['auto', 'sqrt', 'log2']  # Número de características consideradas en cada división
# }

# # búsqueda de cuadrícula
# model=GridSearchCV(pipeline,
#                    param_grid=param_grid,
#                    cv=6,
#                    scoring='neg_mean_squared_error',
#                    n_jobs=-1,
#                    refit=True)

In [ ]:
# partición de la data en conjunto de entrenamiento y prueba
x_train, x_test, y_train, y_test=train_test_split(
    project_2025.drop(columns=['Cantidad Real Gastada']),
    project_2025['Cantidad Real Gastada'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# # ajuste del modelo
# model.fit(x_train, y_train)
# print("Best score: ", model.best_score_)
# print("Best parameters: ", model.best_params_)

In [ ]:
# # alamacenamiento del modelo
# model_filename='files/models'
# os.makedirs(model_filename,exist_ok=True)

# model_path=os.path.join(model_filename,'XGBoost.pkl.gz')
# with gzip.open(model_path,'wb') as file:
#     pickle.dump(model,file)

In [ ]:
model_path='files/models/XGBoost.pkl.gz'

In [ ]:
# caculo de metricas del modelo
metrics_calculate_from_file(model_path, x_train, x_test, y_train, y_test,
                            output_path='files/output/metrics3.json')

In [ ]:
# Grafico de ajuste del modelo
plot_model_predictions(model_path, x_test, y_test)

In [ ]:
# importancia de las variables del modelo
plot_feature_importance_from_file(model_path)

#### MLP Regressor


In [2]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    median_absolute_error)

In [22]:
# lectura de la data
project_2025=pd.read_csv('Archivos/Data/DataFrame.csv')

In [ ]:
# preprocesamiento de la data
numerical_features=['Gasto a la Fecha Estimado', 'Presupuesto Final Estimado',
                    'Presupuesto del Proyecto','Probabilidad','Impacto','Duración Proyectada',
                    'Desviación Presupuestaria']
categorical_features=['Tipo Proyecto','Fase del Proyecto']

preprocessor=ColumnTransformer(
    transformers=
    [('num', StandardScaler(),numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# creación de la pipeline
steps=[
    ('preprocessor', preprocessor),
    ('regressor', MLPRegressor())
]
pipeline=Pipeline(steps)

In [ ]:
# hiperparámetros para la búsqueda de cuadrícula
param_grid=param_grid={
    'regressor__hidden_layer_sizes': [(100,), (50, 50)],  # Una capa de 100 o dos de 50
    'regressor__activation': ['relu', 'tanh'],
    'regressor__solver': ['adam', 'sgd'],
    'regressor__alpha': [0.0001, 0.001, 0.01],
    'regressor__learning_rate': ['constant', 'adaptive']
}

# búsqueda de cuadrícula
model=GridSearchCV(pipeline,
                   param_grid=param_grid,
                   cv=6,
                   scoring='neg_mean_squared_error',
                   n_jobs=-1,
                   refit=True)

In [ ]:
# partición de la data en conjunto de entrenamiento y prueba
x_train, x_test, y_train, y_test=train_test_split(
    project_2025.drop(columns=['Cantidad Real Gastada']),
    project_2025['Cantidad Real Gastada'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# ajuste del modelo
model.fit(x_train, y_train)
print("Best score: ", model.best_score_)
print("Best parameters: ", model.best_params_)

In [ ]:
# alamacenamiento del modelo
model_filename='files/models'
os.makedirs(model_filename,exist_ok=True)

model_path=os.path.join(model_filename,'RedNeuronal.pkl.gz')
with gzip.open(model_path,'wb') as file:
    pickle.dump(model,file)

In [ ]:
model_path='files/models/RedNeuronal.pkl.gz'

In [ ]:
# caculo de metricas del modelo
metrics_calculate_from_file(model_path, x_train, x_test, y_train, y_test,
                            output_path='files/output/metrics4.json')

In [ ]:
# Grafico de ajuste del modelo
plot_model_predictions(model_path, x_test, y_test)

In [ ]:
# importancia de las variables del modelo
plot_feature_importance_from_file(model_path)